In [22]:
# Instalar dependencias necesarias de forma segura
import sys
import subprocess

packages = [
    'pandas',
    'numpy',
    'matplotlib',
    'seaborn',
    'scikit-learn',
    'imbalanced-learn',
    'openpyxl'
]

for pkg in packages:
    try:
        if pkg == 'scikit-learn':
            __import__('sklearn')
        elif pkg == 'imbalanced-learn':
            __import__('imblearn')
        else:
            __import__(pkg)
        print(f"✓ {pkg} ya está instalado")
    except ImportError:
        print(f'Instalando {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
        print(f"✓ {pkg} instalado correctamente")

# Importar librerías
try:
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    from sklearn.preprocessing import StandardScaler, LabelEncoder
    from sklearn.model_selection import train_test_split
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.pipeline import Pipeline
    from sklearn.metrics import classification_report, f1_score, confusion_matrix
    from imblearn.over_sampling import SMOTE
    print("\n✓ Todas las librerías importadas correctamente")
except Exception as e:
    print('Error en importaciones:', e)
    raise

✓ pandas ya está instalado
✓ numpy ya está instalado
✓ matplotlib ya está instalado
✓ seaborn ya está instalado
✓ scikit-learn ya está instalado
✓ imbalanced-learn ya está instalado
✓ openpyxl ya está instalado

✓ Todas las librerías importadas correctamente


In [23]:
import pandas as pd

import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import StandardScaler, LabelEncoder



# 1. Cargar el dataset

# Nota: Asegúrate de que el nombre del archivo coincida con la ruta en tu entorno

df = pd.read_excel("Dry_Bean_Dataset.xlsx")



# Visualizar si hay valores nulos inicialmente

print("Análisis de datos faltantes por columna:")

print(df.isnull().sum())



# 2. Separar características (X) y la variable objetivo (y)

X = df.drop(columns=['Class'])

y = df['Class']



# 3. Codificación de la Variable Categórica (Target)

# Usamos LabelEncoder porque 'Class' es la variable objetivo multiclasificación

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)



# 4. Dividir el dataset en Entrenamiento y Prueba (CRÍTICO para evitar Data Leakage)

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)



# 5. Estrategia de Imputación de Datos Faltantes (Missing Data)

# Aunque el dataset de Dry Beans no suele tener nulos, es una buena práctica incluir un imputador.

# Usamos la mediana porque es menos sensible a outliers en características geométricas.

imputer = SimpleImputer(strategy='median')



# IMPORTANTE: Hacemos fit_transform SOLAMENTE en el set de entrenamiento

X_train_imputed = imputer.fit_transform(X_train)

# Para el test, solo hacemos transform (usando las medianas calculadas en el train)

X_test_imputed = imputer.transform(X_test)



# 6. Estandarización / Normalización

# Usamos StandardScaler para que las variables tengan media 0 y desviación estándar 1.

# Esto es vital porque 'Area' está en decenas de miles y los 'ShapeFactors' son decimales muy pequeños.

scaler = StandardScaler()



# Nuevamente, fit_transform en Train, y solo transform en Test

X_train_scaled = scaler.fit_transform(X_train_imputed)

X_test_scaled = scaler.transform(X_test_imputed)



# Convertir de nuevo a DataFrames (opcional, para mejor visualización o manejo posterior)

feature_names = X.columns

X_train_final = pd.DataFrame(X_train_scaled, columns=feature_names)

X_test_final = pd.DataFrame(X_test_scaled, columns=feature_names)



print("\nPreprocesamiento completado con éxito.")

print(f"Dimensiones de X_train_final: {X_train_final.shape}")

print(f"Dimensiones de X_test_final: {X_test_final.shape}")

Análisis de datos faltantes por columna:
Area               0
Perimeter          0
MajorAxisLength    0
MinorAxisLength    0
AspectRation       0
Eccentricity       0
ConvexArea         0
EquivDiameter      0
Extent             0
Solidity           0
roundness          0
Compactness        0
ShapeFactor1       0
ShapeFactor2       0
ShapeFactor3       0
ShapeFactor4       0
Class              0
dtype: int64

Preprocesamiento completado con éxito.
Dimensiones de X_train_final: (10888, 16)
Dimensiones de X_test_final: (2723, 16)


In [24]:
# Importar las librerías de balanceo
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from collections import Counter

# Visualizar el conteo original en el Train Set
print(f"Distribución original en y_train: {Counter(y_train)}")

# ---------------------------------------------------------
# TÉCNICA 1: Random Undersampling (Submuestreo)
# ---------------------------------------------------------
# Reduce todas las clases al tamaño de la clase minoritaria (BOMBAY: 418)
rus = RandomUnderSampler(random_state=42)
X_train_rus, y_train_rus = rus.fit_resample(X_train_final, y_train)
print(f"Distribución tras UnderSampling: {Counter(y_train_rus)}")

# ---------------------------------------------------------
# TÉCNICA 2: Random Oversampling (Sobremuestreo)
# ---------------------------------------------------------
# Duplica los registros de las clases minoritarias hasta igualar a la mayoritaria (DERMASON: 2837)
ros = RandomOverSampler(random_state=42)
X_train_ros, y_train_ros = ros.fit_resample(X_train_final, y_train)
print(f"Distribución tras OverSampling: {Counter(y_train_ros)}")

# ---------------------------------------------------------
# TÉCNICA 3: SMOTE (Synthetic Minority Over-sampling Technique)
# ---------------------------------------------------------
# Crea muestras SINTÉTICAS (no duplicadas) para las clases minoritarias
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_final, y_train)
print(f"Distribución tras SMOTE: {Counter(y_train_smote)}")

# Nota: El conjunto de prueba (X_test_final, y_test) se mantiene INTACTO para evaluar el modelo con datos reales.

Distribución original en y_train: Counter({np.int64(3): 2837, np.int64(6): 2109, np.int64(5): 1621, np.int64(4): 1542, np.int64(2): 1304, np.int64(0): 1057, np.int64(1): 418})
Distribución tras UnderSampling: Counter({np.int64(0): 418, np.int64(1): 418, np.int64(2): 418, np.int64(3): 418, np.int64(4): 418, np.int64(5): 418, np.int64(6): 418})
Distribución tras OverSampling: Counter({np.int64(5): 2837, np.int64(4): 2837, np.int64(2): 2837, np.int64(6): 2837, np.int64(3): 2837, np.int64(0): 2837, np.int64(1): 2837})
Distribución tras SMOTE: Counter({np.int64(5): 2837, np.int64(4): 2837, np.int64(2): 2837, np.int64(6): 2837, np.int64(3): 2837, np.int64(0): 2837, np.int64(1): 2837})


In [25]:
# Unir X e y para guardarlos cómodamente en CSVs
# 1. Conjunto de Entrenamiento Balanceado (SMOTE) - EL PRINCIPAL
train_smote_df = pd.DataFrame(X_train_smote, columns=feature_names)
train_smote_df['Class'] = y_train_smote
train_smote_df.to_csv("train_data_SMOTE.csv", index=False)

# 2. Conjunto de Entrenamiento Original (Desbalanceado) - PARA COMPARAR EN EL PUNTO 5
train_original_df = X_train_final.copy()
train_original_df['Class'] = y_train
train_original_df.to_csv("train_data_IMBALANCED.csv", index=False)

# 3. Conjunto de Prueba (Test) - INTACTO, SOLO PARA EVALUACIÓN FINAL
test_df = X_test_final.copy()
test_df['Class'] = y_test
test_df.to_csv("test_data_FINAL.csv", index=False)

print("Archivos exportados con éxito para el resto del equipo.")

Archivos exportados con éxito para el resto del equipo.


In [ ]:
# Los datos ya están cargados en la celda anterior.
# Esta celda puede usarse para exploración adicional si es necesario.
print("✓ Dataset completamente procesado y listo para modelado.")
print(f"  - Datos de entrenamiento después de SMOTE: {X_resampled.shape}")
print(f"  - Clases balanceadas: {np.unique(y_resampled)}")

In [ ]:
# =====================================================================
# APARTADO 3: ANÁLISIS EXPLORATORIO ADICIONAL
# =====================================================================
print("\n--- 3. ANÁLISIS EXPLORATORIO DE DATOS ---")

# Visualización de desbalanceo en datos resampled
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
sns.countplot(x=y, palette='Set2')
plt.title('Dataset Original: Desbalanceo de Clases')
plt.xlabel('Clase')
plt.ylabel('Cantidad')
plt.xticks(rotation=45)

plt.subplot(1, 2, 2)
sns.countplot(x=y_resampled_labels, palette='Set2')
plt.title('Después de SMOTE: Clases Balanceadas')
plt.xlabel('Clase')
plt.ylabel('Cantidad')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

# Histogramas de distribuciones de características
print("\nDistribuciones de características (primeras 4 características):")
X_scaled_df.iloc[:, :4].hist(figsize=(12, 5), bins=30)
plt.tight_layout()
plt.show()

# Matriz de correlaciones
print("\nMatriz de correlación de características:")
plt.figure(figsize=(10, 8))
sns.heatmap(X.corr(), annot=False, cmap='coolwarm', square=True)
plt.title('Correlación entre características')
plt.tight_layout()
plt.show()

print("\n✓ Análisis exploratorio completado")

In [ ]:
# =====================================================================
# APARTADO 4: DIVISION EN CONJUNTOS DE ENTRENAMIENTO Y PRUEBA
# =====================================================================
print("\n--- 4. DIVISIÓN TRAIN/TEST ---")

# División estratificada para mantener proporción de clases
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Tamaño del conjunto de entrenamiento: {X_train.shape}")
print(f"Tamaño del conjunto de prueba: {X_test.shape}")

# Estandarización de características
scaler_final = StandardScaler()
X_train_scaled = scaler_final.fit_transform(X_train)
X_test_scaled = scaler_final.transform(X_test)

print("\n✓ Escalado completado")
print(f"  - Media de características en train (primera característica): {X_train_scaled[:, 0].mean():.4f}")
print(f"  - Desv. Estándar en train (primera característica): {X_train_scaled[:, 0].std():.4f}")

array([[-0.3713618 , -0.53113068, -0.69536968, ...,  1.11808237,
         1.48524952,  0.95141553],
       [ 0.02895235,  0.49881788,  0.78351767, ..., -1.32149747,
        -1.87953412,  0.15816127],
       [ 0.72707783,  0.79500482,  0.82438155, ..., -0.7876742 ,
        -0.2375073 , -0.35302543],
       ...,
       [ 0.43664179,  0.87746349,  0.47987974, ..., -0.53535794,
        -0.01143779, -1.4807801 ],
       [ 0.01662606,  0.31205264,  0.64565274, ..., -1.18140133,
        -1.61703817, -0.74823181],
       [ 4.01901639,  3.47586447,  3.51587226, ..., -1.68315117,
        -0.80825296, -0.89219251]], shape=(10888, 16))

In [ ]:
# =====================================================================
# APARTADO 5: MODELADO - MÉTODO A: RandomForest + class_weight
# =====================================================================
print("\n--- 5. MODELO A: RandomForest con class_weight='balanced' ---")

# Crear y entrenar RandomForest
rf_a = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

print("Entrenando RandomForest (Método A)...")
rf_a.fit(X_train_scaled, y_train)
y_pred_a = rf_a.predict(X_test_scaled)

print("\n=== RESULTADOS: RandomForest + class_weight='balanced' ===")
print(classification_report(y_test, y_pred_a))
print("\nMacro F1 Score:", f1_score(y_test, y_pred_a, average="macro"))

print("\nMatriz de Confusión:")
cm_a = confusion_matrix(y_test, y_pred_a)
print(cm_a)

print("\n✓ Modelo A completado")

C:\Users\alex.iturgaitz\AppData\Roaming\Python\Python312\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


=== RandomForest + class_weight='balanced' ===
              precision    recall  f1-score   support

    BARBUNYA       0.94      0.89      0.91       265
      BOMBAY       1.00      1.00      1.00       104
        CALI       0.94      0.94      0.94       326
    DERMASON       0.91      0.92      0.92       709
       HOROZ       0.97      0.95      0.96       386
       SEKER       0.94      0.96      0.95       406
        SIRA       0.86      0.85      0.85       527

    accuracy                           0.92      2723
   macro avg       0.93      0.93      0.93      2723
weighted avg       0.92      0.92      0.92      2723

Macro F1: 0.9326187597475176

Matriz de confusión:
[[236   0  17   0   1   2   9]
 [  0 104   0   0   0   0   0]
 [ 10   0 305   0   5   2   4]
 [  0   0   0 655   0  10  44]
 [  1   0   4   4 368   0   9]
 [  1   0   0   6   0 389  10]
 [  4   0   0  55   7  11 450]]


In [ ]:
# =====================================================================
# APARTADO 6: MODELADO - MÉTODO B: SMOTE + RandomForest en Pipeline
# =====================================================================
print("\n--- 6. MODELO B: Pipeline con SMOTE + RandomForest ---")

# Crear pipeline con SMOTE + RandomForest
smote_pipe = SMOTE(random_state=42)

rf_b = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

pipe_b = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("smote", smote_pipe),
    ("rf", rf_b),
])

print("Entrenando Pipeline (Método B)...")
pipe_b.fit(X_train, y_train)
y_pred_b = pipe_b.predict(X_test)

print("\n=== RESULTADOS: SMOTE + RandomForest en Pipeline ===")
print(classification_report(y_test, y_pred_b))
print("\nMacro F1 Score:", f1_score(y_test, y_pred_b, average="macro"))

print("\nMatriz de Confusión:")
cm_b = confusion_matrix(y_test, y_pred_b)
print(cm_b)

print("\n✓ Modelo B completado")

# =====================================================================
# COMPARACIÓN DE MÉTODOS
# =====================================================================
print("\n--- COMPARACIÓN DE MÉTODOS ---")
f1_a = f1_score(y_test, y_pred_a, average="macro")
f1_b = f1_score(y_test, y_pred_b, average="macro")

print(f"\nMétodo A (class_weight):  F1 Macro = {f1_a:.4f}")
print(f"Método B (SMOTE):         F1 Macro = {f1_b:.4f}")
print(f"Mejor método: {'A (class_weight)' if f1_a > f1_b else 'B (SMOTE)' if f1_b > f1_a else 'Empate'}")

C:\Users\alex.iturgaitz\AppData\Roaming\Python\Python312\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


=== SMOTE + RandomForest ===
              precision    recall  f1-score   support

    BARBUNYA       0.92      0.90      0.91       265
      BOMBAY       1.00      1.00      1.00       104
        CALI       0.94      0.94      0.94       326
    DERMASON       0.92      0.91      0.91       709
       HOROZ       0.96      0.95      0.96       386
       SEKER       0.94      0.96      0.95       406
        SIRA       0.86      0.86      0.86       527

    accuracy                           0.92      2723
   macro avg       0.93      0.93      0.93      2723
weighted avg       0.92      0.92      0.92      2723

Macro F1: 0.9325596766874037

Matriz de confusión:
[[239   0  15   0   1   3   7]
 [  0 104   0   0   0   0   0]
 [ 12   0 305   0   5   2   2]
 [  0   0   0 647   1  12  49]
 [  2   0   5   4 368   0   7]
 [  1   0   0   5   0 390  10]
 [  6   0   0  50   7  10 454]]
